# 01 · Truy vết nhãn và giữ split theo bệnh nhân
**Câu hỏi:** từ chú thích SCP, vì sao một ECG nhận vector nhãn này; những ECG nào không đủ mục tiêu học; làm sao ngăn một người xuất hiện ở cả train và test?

**Đầu vào:** nguồn xác minh ở 00. **Đầu ra:** `metadata.csv` với chú thích gốc, nhãn và fold. **Tiếp theo:** 02 QC tín hiệu.

| Vị trí | Nhãn | Nhóm chẩn đoán ECG |
|---|---|---|
| 0 | NORM | Bình thường theo chú thích |
| 1 | MI | Nhồi máu cơ tim |
| 2 | STTC | Thay đổi ST/T |
| 3 | CD | Rối loạn dẫn truyền |
| 4 | HYP | Phì đại |

Đây là multi-label: MI và CD tạo `[0,1,0,1,0]`. NORM không chứng minh không suy tim; MI/HYP không phải nhãn suy tim.

**Trạng thái bản sửa 01/11/2026:** source giữ sạch output; các hình/bảng chỉ có sau khi Line chạy trên máy đích. Output lưu ngày 24/10/2026 thuộc phiên bản notebook cũ, không chứng minh các cell mới đã chạy. Các cell ghi “TOÀN BỘ” có thể đọc/ghi nhiều GB; không chạy chỉ để xem nội dung.

In [ ]:
from pathlib import Path
import os
import sys

def tim_thu_muc_project():
    explicit = os.environ.get("PTBXL_PROJECT_ROOT")
    candidates = [Path(explicit).expanduser().resolve()] if explicit else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / "ptbxl_pipeline").is_dir() and (candidate / "requirements-ptbxl.txt").is_file():
            return candidate.resolve()
    raise FileNotFoundError("Không tìm thấy project. Đặt PTBXL_PROJECT_ROOT tới thư mục chứa ptbxl_pipeline và requirements-ptbxl.txt.")

ROOT = tim_thu_muc_project()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
RAW_DIR = Path(os.environ.get("PTBXL_RAW_DIR", str(ROOT / "data/raw/ptbxl/1.0.3"))).expanduser().resolve()
CACHE_DIR = ROOT / "data/cache"
OUTPUT_DIR = Path(os.environ.get("PTBXL_OUTPUT_DIR", str(ROOT / "data/processed/ptbxl_v1.0.3_500hz_minimal"))).expanduser().resolve()
REPORT_DIR = ROOT / "artifacts/ptbxl_reports" / OUTPUT_DIR.name
TARGET_FS = 500
PREPROCESSING = "minimal"
EXPECTED_RECORDS = 21799
print("Project:", ROOT)
print("Dữ liệu gốc:", RAW_DIR)
print("Bộ xuất:", OUTPUT_DIR)

In [ ]:
from ptbxl_pipeline.core import PipelineConfig

config = PipelineConfig(
    raw_dir=RAW_DIR,
    output_dir=OUTPUT_DIR,
    target_fs=TARGET_FS,
    preprocessing=PREPROCESSING,
    expected_records=EXPECTED_RECORDS,
)

## 1. Tạo manifest: chính sách ánh xạ có thể kiểm tra
Lấy các **mã hiện diện** trong `scp_codes`, chỉ ánh xạ những mã có `diagnostic == 1` qua `diagnostic_class`. Likelihood 0 biểu thị chưa biết độ tin cậy; không tự xem là chẩn đoán âm. Giữ mã chưa biết để rà lại, không tự tạo thêm lớp.

Cell này ghi metadata và provenance, đồng thời kiểm tra các file nguồn cần dùng. Có thể mất thời gian với toàn bộ dữ liệu. Cấu hình/nguồn/runtime khác release cũ phải dùng output mới.

In [ ]:
from IPython.display import display
from ptbxl_pipeline.core import CLASSES, build_manifest

metadata = build_manifest(config)
label_columns = [f"y_{name}" for name in CLASSES]
assert len(metadata) == EXPECTED_RECORDS and metadata.ecg_id.is_unique
assert metadata[label_columns].isin([0, 1]).all().all()
train_all = metadata.loc[metadata.split.eq("train")].sort_values("ecg_id").copy()
train_labeled = train_all.loc[train_all.has_label.eq(1)].copy()
print("Manifest:", OUTPUT_DIR / "metadata.csv")

## 2. Truy vết từng mã của các ví dụ train thực
Hàm này là phần giải thích độc lập cho người đọc; kết quả phải khớp nhãn do core xuất. Chọn một ECG đa nhãn, một ECG có mã likelihood 0, một ca NORM đi cùng bất thường và một ca không có nhãn nếu các nhóm tồn tại. Không ép tạo ví dụ giả khi nhóm vắng mặt.

In [ ]:
import json
import numpy as np
import pandas as pd

def trace_scp_codes(scp_codes_json, statements):
    """Giải thích ánh xạ theo sự hiện diện; không dùng likelihood làm ngưỡng."""
    classes = ("NORM", "MI", "STTC", "CD", "HYP")
    annotations = json.loads(scp_codes_json)
    rows, labels = [], set()
    for key, likelihood in sorted(annotations.items()):
        known = key in statements.index
        diagnostic = bool(known and statements.at[key, "diagnostic"] == 1)
        superclass = statements.at[key, "diagnostic_class"] if diagnostic else ""
        mapped = diagnostic and superclass in classes
        if mapped:
            labels.add(superclass)
        rows.append({"ma_SCP": key, "likelihood": likelihood, "co_trong_tu_dien": known,
                     "diagnostic": diagnostic, "superclass": superclass if mapped else "",
                     "duoc_anh_xa": mapped})
    columns = ["ma_SCP", "likelihood", "co_trong_tu_dien", "diagnostic", "superclass", "duoc_anh_xa"]
    return pd.DataFrame(rows, columns=columns), np.asarray([int(name in labels) for name in classes])

In [ ]:
statements = pd.read_csv(RAW_DIR / "scp_statements.csv", index_col=0)
zero_likelihood = train_all.scp_codes_json.map(lambda value: any(v == 0 for v in json.loads(value).values()))
unknown_present = train_all.unknown_scp_codes_json.map(lambda value: len(json.loads(value)) > 0)
example_groups = {
    "đa nhãn": train_all[label_columns].sum(axis=1).gt(1),
    "có mã likelihood 0": zero_likelihood,
    "NORM đi cùng bất thường": train_all.norm_with_abnormal.eq(1),
    "không có diagnostic superclass": train_all.has_label.eq(0),
    "mã chưa biết": unknown_present,
}
for reason, mask in example_groups.items():
    candidates = train_all.loc[mask]
    if candidates.empty:
        print(reason, ": không có ví dụ trong train")
        continue
    row = candidates.iloc[0]
    trace, vector = trace_scp_codes(row.scp_codes_json, statements)
    np.testing.assert_array_equal(vector, row[label_columns].to_numpy(dtype=int))
    print(f"ECG {int(row.ecg_id)} — {reason} — {list(CLASSES)} → {vector.tolist()}")
    display(trace)

**Cách đọc:** mã form/rhythm được giữ ở metadata nhưng không nhất thiết đóng góp vào 5 lớp diagnostic. “Không ánh xạ” khác “âm tính tất cả bệnh”. Một vector toàn 0 vì không có diagnostic superclass không được đổi thành NORM. Cờ NORM+bất thường chỉ yêu cầu truy vết, chưa đủ cơ sở để xóa một nhãn.

Bảng tiếp theo có mẫu số là **toàn bộ train trước QC**; các nhóm có thể chồng nhau nên không cộng để suy số ECG duy nhất.

In [ ]:
quality_flags = pd.Series({
    "khong_co_nhan_diagnostic": int(train_all.has_label.eq(0).sum()),
    "NORM_kem_bat_thuong": int(train_all.norm_with_abnormal.eq(1).sum()),
    "co_ma_likelihood_0": int(zero_likelihood.sum()),
    "co_ma_chua_biet": int(unknown_present.sum()),
}, name="so_ECG")
flag_table = quality_flags.to_frame()
flag_table["mau_so_train_truoc_QC"] = len(train_all)
flag_table["ty_le_pct"] = 100 * flag_table.so_ECG / len(train_all)
display(flag_table)

## 3. Kiểm tra cấu trúc split trên cả ba tập
Giữ fold 1–8 train, 9 validation, 10 test. Một bệnh nhân có thể có nhiều ECG; chia ngẫu nhiên theo ECG có thể làm rò rỉ người bệnh. Đây là QA cấu trúc toàn bộ split, không phải khai thác test để chọn phép xử lý.

In [ ]:
expected_split = metadata.strat_fold.map(lambda fold: "train" if 1 <= int(fold) <= 8 else "validation" if int(fold) == 9 else "test")
assert metadata.strat_fold.isin(range(1, 11)).all()
assert metadata.split.eq(expected_split).all()
assert metadata.groupby("patient_id").split.nunique().max() == 1
patient_sets = {split: set(metadata.loc[metadata.split.eq(split), "patient_id"]) for split in ("train", "validation", "test")}
for left, right in (("train", "validation"), ("train", "test"), ("validation", "test")):
    assert not patient_sets[left] & patient_sets[right], f"Giao bệnh nhân: {left}/{right}"
display(metadata.groupby("split").agg(so_ECG=("ecg_id", "size"), so_benh_nhan=("patient_id", "nunique"), co_nhan=("has_label", "sum")).reindex(["train", "validation", "test"]))
display(train_all.groupby("patient_id").size().value_counts().sort_index().rename_axis("so_ECG_moi_nguoi").to_frame("so_benh_nhan_train"))

## 4. Mất cân bằng và đồng xuất hiện trên train có nhãn
Mẫu số ở đây là train có nhãn **trước QC tín hiệu**. Prevalence là tỷ lệ ECG mang từng nhãn; tổng có thể lớn hơn 100%. Cardinality đếm nhãn trên mỗi ECG. Ô (MI,STTC) của ma trận là số ECG mang đồng thời hai nhãn, không chứng minh quan hệ nhân quả.

In [ ]:
import matplotlib.pyplot as plt

label_matrix = train_labeled[label_columns].to_numpy(dtype=np.int64)
prevalence = pd.DataFrame({"nhan": CLASSES, "so_duong": label_matrix.sum(axis=0), "so_am": len(train_labeled) - label_matrix.sum(axis=0)})
prevalence["ty_le_duong_pct"] = 100 * prevalence.so_duong / len(train_labeled)
display(prevalence)
cardinality = train_labeled[label_columns].sum(axis=1).value_counts().sort_index()
cooccurrence = pd.DataFrame(label_matrix.T @ label_matrix, index=CLASSES, columns=CLASSES)
display(cooccurrence)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].bar(prevalence.nhan, prevalence.ty_le_duong_pct)
axes[0].set_ylabel("% train có nhãn, trước QC")
axes[0].set_title("Prevalence")
axes[1].bar(cardinality.index, cardinality.values)
axes[1].set_xlabel("Số nhãn/ECG")
axes[1].set_ylabel("Số ECG")
axes[1].set_title("Cardinality")
im = axes[2].imshow(cooccurrence, cmap="Blues")
axes[2].set_xticks(range(5), CLASSES)
axes[2].set_yticks(range(5), CLASSES)
axes[2].set_title("Số ECG đồng xuất hiện")
fig.colorbar(im, ax=axes[2])
fig.tight_layout()
plt.show()

## 5. Thiếu thông tin nhân khẩu và giới hạn mô tả
Khảo sát **train trước QC**, theo ECG (một người có thể lặp lại); không gọi đây là phân bố của quần thể bệnh nhân độc lập. Tuổi bị ẩn danh ở nhóm cao tuổi: mã lớn như 300+ không được đem tính tuổi trung bình như tuổi thật. Bảng đếm mã thô vẫn được giữ để kiểm tra; biểu đồ chỉ hiển thị tuổi 0–89 và ghi rõ phần bị bỏ khỏi biểu đồ. Không điền chiều cao/cân nặng thiếu bằng 0.

In [ ]:
demographic_fields = [name for name in ("age", "sex", "height", "weight", "device", "site") if name in train_all]
missing_rows = []
for name in demographic_fields:
    series = train_all[name]
    missing = series.isna() | series.astype(str).str.strip().str.lower().isin(["", "nan", "none"])
    missing_rows.append({"cot": name, "thieu": int(missing.sum()), "mau_so_ECG_train": len(train_all), "ty_le_thieu_pct": 100 * missing.mean()})
display(pd.DataFrame(missing_rows))
age = pd.to_numeric(train_all.age, errors="coerce")
age_visible = age.between(0, 89)
display(pd.Series({"tuoi_0_89": int(age_visible.sum()), "ma_tuoi_300_tro_len": int(age.ge(300).sum()), "gia_tri_khac_khong_thieu": int((age.notna() & ~age_visible & ~age.ge(300)).sum()), "thieu_hoac_khong_doc_duoc": int(age.isna().sum())}).to_frame("so_ECG_train"))
display(train_all.sex.astype(str).replace("", "<thiếu>").value_counts(dropna=False).rename_axis("ma_sex_nguon").to_frame("so_ECG_train"))
fig, axis = plt.subplots(figsize=(8, 3))
axis.hist(age.loc[age_visible], bins=np.arange(0, 95, 5))
axis.set(xlabel="Tuổi hiển thị 0–89", ylabel="Số ECG train", title=f"Tuổi có thể mô tả trực tiếp: {int(age_visible.sum())}/{len(train_all)} ECG; không gồm mã tuổi ẩn danh")
plt.show()

## Quyết định và điểm cần theo dõi
Giữ 5 lớp, chính sách theo sự hiện diện SCP và fold chính thức. Ghi lại ca không nhãn, NORM đi cùng bất thường và mã chưa biết; không âm thầm sửa nhãn. Chưa cân bằng dữ liệu bằng cách xóa/nhân bản ECG; weighting/sampling thuộc thí nghiệm train sau này.

Các bảng trên chưa phải cohort cuối: 02 còn kiểm tra tín hiệu; 04 đối chiếu số lượng và ảnh hưởng loại theo nhãn. Nếu đổi quy tắc nhãn, phải có lý do, release mới và so sánh riêng; không chỉnh theo kết quả test.

Nguồn: [PTB-XL](https://physionet.org/content/ptb-xl/1.0.3/) (đối chiếu 01/11/2026); [benchmark ánh xạ](https://github.com/helme/ecg_ptbxl_benchmarking/blob/master/code/utils/utils.py) (đã đối chiếu 24/10/2026). Tiếp theo: **02_tien_xu_ly_va_qc.ipynb**.